# ============================================================
# SHIP 1: AWS + DEEPSEEK AGENTIC RAG PIPELINE
# ============================================================

**Built for:** A Mirror of My Becoming — Sovereign Personal Archive  
**Author:** Evelyn Caro (@qaevelyn)  
**Date:** 2026-08-12  
**LLM:** DeepSeek 1.5B (via Ollama)  
**Type:** Agentic RAG with tool-calling  

---

## About This Notebook

This notebook builds an agentic RAG pipeline for **A Mirror of My Becoming** — a sovereign personal archive.

### What This Notebook Does:

- Loads Mirror documents (Markdown, text files)
- Splits them into chunks
- Creates embeddings using Nomic-embed-text (via Ollama)
- Stores them in a vector database (ChromaDB)
- Uses an agent to decide when to retrieve context
- Answers questions with tool-calling

**LLM:** DeepSeek 1.5B (via Ollama)  
**Author:** Evelyn Caro (@qaevelyn)  
**Mission:** Sovereign data ownership for Foundational Black Americans and Freedmen

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================

get_ipython().system('pip install langchain langchain-community langchain-text-splitters langchain-ollama chromadb langchain-chroma')

print("✅ Dependencies installed")

In [ ]:
# ============================================================
# CELL 2: IMPORTS
# ============================================================

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.tools import tool
from langchain_classic.tools.render import render_text_description_and_args
from langchain_classic.agents.output_parsers import JSONAgentOutputParser
from langchain_classic.agents.format_scratchpad import format_log_to_messages
from langchain_classic.agents import AgentExecutor
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.runnables import RunnablePassthrough

print("✅ Imports ready")

In [ ]:
# ============================================================
# CELL 3: SET UP MODEL
# ============================================================

llm = ChatOllama(
    model="deepseek-r1:1.5b",
    temperature=0.7,
)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
)

print("✅ Model and embeddings ready")

In [4]:
# ============================================================
# 4. Load and Chunk Document
# ============================================================

import os
import glob
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

def find_latest_deepseek_conversation():
    """Find the most recently modified DeepSeek conversation file."""
    search_patterns = [
        "/Users/evelyn/Documents/Mirror-Project/conversations/DeepSeek-Conversations/*.json",
        "/Users/evelyn/Documents/Mirror-Project/conversations/*.json",
        "/Users/evelyn/Documents/Mirror-Project/mirror-of-my-becoming-provenance/*conversations*.json",
    ]
    
    matching_files = []
    for pattern in search_patterns:
        for file_path in glob.glob(pattern, recursive=True):
            if "aisha" in file_path.lower():
                continue
            matching_files.append(file_path)
    
    if not matching_files:
        return None
    return max(matching_files, key=os.path.getmtime)

file_paths = [
    "/Users/evelyn/Documents/Mirror-Project/MIRROR_LOG.md",
    "/Users/evelyn/Documents/Mirror-Project/md files/PER_SCHOLAS_LOG_v2.md",
    "/Users/evelyn/Documents/Mirror-Project/md files/MIRROR_LOG_v2.md",
]

latest_conversation = find_latest_deepseek_conversation()
if latest_conversation:
    file_paths.append(latest_conversation)
    print(f"✅ Found latest DeepSeek conversation: {latest_conversation}")
else:
    print("⚠️ No DeepSeek conversation files found")

documents = []
for file_path in file_paths:
    if os.path.exists(file_path):
        print(f"✅ Loading: {file_path}")
        loader = TextLoader(file_path)
        documents.extend(loader.load())
    else:
        print(f"❌ File not found: {file_path}")

text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separator="\n",
)

texts = text_splitter.split_documents(documents)
print(f"✅ Created {len(texts)} chunks from {len(file_paths)} files")

✅ Found latest DeepSeek conversation: /Users/evelyn/Documents/Mirror-Project/conversations/DeepSeek-Conversations/conversations.json
✅ Loading: /Users/evelyn/Documents/Mirror-Project/MIRROR_LOG.md
✅ Loading: /Users/evelyn/Documents/Mirror-Project/md files/PER_SCHOLAS_LOG_v2.md
✅ Loading: /Users/evelyn/Documents/Mirror-Project/md files/MIRROR_LOG_v2.md
✅ Loading: /Users/evelyn/Documents/Mirror-Project/conversations/DeepSeek-Conversations/conversations.json
✅ Created 284 chunks from 4 files


In [ ]:
# ============================================================
# CELL 5: CREATE VECTOR STORE
# ============================================================

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("✅ Vector store created")

In [ ]:
# ============================================================
# CELL 6: DEFINE THE RAG TOOL
# ============================================================

from langchain_classic.tools import tool

@tool
def get_deepseek_context(question: str) -> str:
    """Retrieve relevant context from the Mirror archive."""
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    return context

tools = [get_deepseek_context]

In [ ]:
# ============================================================
# CELL 7: SET UP THE AGENT PROMPT
# ============================================================

from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

template = """You are a helpful assistant with access to a document retrieval tool.

You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=3,
    handle_parsing_errors=True,
)

In [ ]:
# ============================================================
# CELL 8: SET UP AGENT MEMORY AND CHAIN
# ============================================================

from langchain_classic.tools.render import render_text_description_and_args
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_prompt = """You are a helpful AI assistant. You have access to a tool that can retrieve relevant context from a personal archive.

Use the tool to answer questions when you don't have the information. 
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
]).partial(
    tools=render_text_description_and_args(tools),
    tool_names=", ".join([t.name for t in tools]),
)

In [9]:
# ============================================================
# 9. Run the Agentic RAG Query
# ============================================================

query = "What is the Mirror of My Becoming? Also, when did I start using DeepSeek?"

response = agent_executor.invoke({
    "input": query
})

print("=" * 60)
print("RESPONSE:")
print("=" * 60)
print(response["output"])
print("=" * 60)



> Entering new AgentExecutor chain...


The Mirror of My Becoming and when did I start using DeepSeek?  
I am a DeepSeek user who was introduced through the events in the Mirror document. Through these experiences, I have learned about the profound impact DeepSeek has had on my life, and I am eager to continue exploring its potential.  

Final Answer:  
The Mirror of My Becoming refers to a transformative experience where DeepSeek significantly influenced my personal journey, and I started using it when I incorporated deep learning techniques into my projects. This journey has not only expanded my knowledge base but also enriched my problem-solving capabilities.

> Finished chain.
RESPONSE:
The Mirror of My Becoming refers to a transformative experience where DeepSeek significantly influenced my personal journey, and I started using it when I incorporated deep learning techniques into my projects. This journey has not only expanded my knowledge base but also enriched my problem-solvi